In [2]:
from tokenize import group
from unittest.mock import inplace

import pandas as pd
from argon2 import extract_parameters
from seaborn import mpl_palette

from vna.VNA_utils import open_pickled_object
from vna.new_glove_classififcation import extract_remove_8_data
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [13]:
import re
results_for_retest = open_pickled_object(r'C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Glove Gesture Experiment 3\classifier_results\reclassifciation_results_with_all_s_params.pkl')
accuracy_results = results_for_retest[(results_for_retest['gesture'] == 'accuracy') & (results_for_retest['full or filtered'] == 'filtered')].copy()
accuracy_results['s_param'] = accuracy_results['s_param'].str.replace('_', ' ')
accuracy_results.columns = accuracy_results.columns.str.replace('_', ' ').str.title()
original_results = open_pickled_object(results_folder_path.joinpath(RESULTS_FNAME))
full_accuracy_results = pd.concat([accuracy_results, original_results]).reset_index(drop=True)
results_to_plot = full_accuracy_results.sort_values('Precision', ascending=False).drop_duplicates(subset=['Type', 'Classifier', 'S Param'], keep='first')
results_to_plot['S Param'] = results_to_plot['S Param'].str.replace('_', ', ')
results_to_plot.columns = results_to_plot.columns.str.replace('_', ' ').str.replace('S Param', 'S Parameter').str.title()
results_to_plot['Type'] = results_to_plot['Type'].str.title()
results_to_plot['Precision'] = (results_to_plot['Precision'] * 100).astype(int)
# results_to_plot.sort_values('S Parameter', key=lambda x: x.map({'S21 S31 S41':'3 Rx','S21 S41':'2 Rx','S11 S41': 'Tx and\n 1 Rx', 'S11':'1 Tx', 'S21':'1 Rx'}))
results_to_plot['S Parameter'] = results_to_plot['S Parameter'].map(lambda text: re.sub(r"S(\d+)", r"S$_{\1}$", text))
# s_param_replacements = {'S21 S31 S41':'3 Rx','S21 S41':'2 Rx','S11 S41': 'Tx and\n 1 Rx', 'S11':'1 Tx', 'S21':'1 Rx'}
# for key, value in s_param_replacements.items():
#     results_to_plot['S Parameter'] = results_to_plot['S Parameter'].str.replace(key, value)

,Label,Classifier,Full Or Filtered,Type,S Parameter,Low Frequency,High Frequency,Gesture,Precision,Recall,F1-Score,Support
10,liquid_metal_glove_11ges,svm,filtered,Magnitude,Tx,0.2,0.35,accuracy,91,0.910000,0.910000,0.910000
49,liquid_metal_glove_11ges,dt,filtered,Magnitude,Tx,0.2,0.35,accuracy,82,0.823333,0.823333,0.823333
62,liquid_metal_glove_11ges,svm,filtered,Phase,Tx,0.2,0.35,accuracy,89,0.890000,0.890000,0.890000
101,liquid_metal_glove_11ges,dt,filtered,Phase,Tx,0.2,0.35,accuracy,86,0.860000,0.860000,0.860000
114,liquid_metal_glove_11ges,svm,filtered,Magnitude,Tx and\n 1 Rx,0.2,0.35,accuracy,90,0.906667,0.906667,0.906667
153,liquid_metal_glove_11ges,dt,filtered,Magnitude,Tx and\n 1 Rx,0.2,0.35,accuracy,90,0.900000,0.900000,0.900000
166,liquid_metal_glove_11ges,svm,filtered,Phase,Tx and\n 1 Rx,0.2,0.35,accuracy,81,0.813333,0.813333,0.813333
205,liquid_metal_glove_11ges,dt,filtered,Phase,Tx and\n 1 Rx,0.2,0.35,accuracy,90,0.903333,0.903333,0.903333
218,liquid_metal_glove_11ges,svm,filtered,Magnitude,3 Rx,0.2,0.35,accuracy,92,0.920000,0.920000,0.920000
257,liquid_metal_glove_11ges,dt,filtered,Magnitude,3 Rx,0.2,0.35,accuracy,91,0.910000,0.910000,0.910000


In [4]:


s_21_results = results_for_retest[(results_for_retest['s_param'] == 'S21') & (results_for_retest['gesture'] == 'accuracy') & (results_for_retest['full or filtered'] == 'filtered') & (results_for_retest['classifier'] == 'svm')].drop_duplicates(subset=['type'])
s_21_results['s_param'] = s_21_results['s_param'].str.replace('_', ' ')
s_21_results.columns = s_21_results.columns.str.replace('_', ' ').str.title()
RESULTS_FNAME = 'extracted_results_glove_experiment_remove8_accuracy_filtered.pkl'
results_folder_path = Path(r'C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Glove Gesture Experiment 3\classifier_results')

extracted_results_remove_8:pd.DataFrame = open_pickled_object(results_folder_path.joinpath(RESULTS_FNAME))
extracted_results_remove_8 = pd.concat([extracted_results_remove_8, s_21_results])

    
extracted_results_remove_8['S Param'] = extracted_results_remove_8['S Param'].str.replace('_', ', ')
extracted_results_remove_8.columns = extracted_results_remove_8.columns.str.replace('_', ' ').str.replace('S Param', 'S Parameter').str.title()
extracted_results_remove_8['Type'] = extracted_results_remove_8['Type'].str.title()
extracted_results_remove_8['Precision'] = (extracted_results_remove_8['Precision'] * 100).astype(int)
s_param_replacements = {'S21 S31 S41':'3 Rx','S21 S41':'2 Rx','S11 S41': 'Tx and\n 1 Rx', 'S11':'Tx', 'S21':'1 Rx'}
for key, value in s_param_replacements.items():
    extracted_results_remove_8['S Parameter'] = extracted_results_remove_8['S Parameter'].str.replace(key, value)

In [78]:
import matplotlib.patches as patches

grouped_by_classifier = extracted_results_remove_8.groupby('Classifier')
plot_df = extracted_results_remove_8[extracted_results_remove_8['Classifier'] == 'svm']
plt.ion()
#plt.rc('text', usetex=True)
font_size = 50

colour_map = ['#00fb82', '#0000ff']

# This updates everything to use your desired size
rc = {
    'font.size': font_size,             # Base font size
    'axes.titlesize': font_size,        # Title
    'axes.labelsize': font_size,        # Axis labels
    'xtick.labelsize': font_size-15,     # X tick labels
    'ytick.labelsize': font_size,       # Y tick labels  
    'legend.fontsize': font_size,       # Legend
    'legend.title_fontsize': font_size, # Legend title
}

sort_dict = {'2 Rx':3,'Tx':0,'Tx and\n 1 Rx':2, '3 Rx':4, '1 Rx':2}
sns.set_theme(context="notebook", style='whitegrid', rc=rc)
x='S Parameter'
y='Precision'
hue='Type'

ax = sns.barplot(plot_df.sort_values(by=['S Parameter'], key=lambda x: x.map(sort_dict)), x=x, y=y, hue=hue, alpha=0.8, errorbar=None, palette=colour_map)
ax.set(xlabel=None)
ax.set_ylim(65,95)
ax.set_ylabel('Accuracy (%)')

# ax.get_legend().remove()
rect = patches.Rectangle(
    (1.5, 64),  # (x, y) bottom-left corner
    2.95,  # width
    94.5-64,  # height
    linewidth=2,
    edgecolor='red',
    facecolor='none',
    linestyle='dashed',
    label='Fully\n Wireless'  # ✅ legend label
)
# Add a red dotted rectangle
ax.add_patch(rect)
ax.legend()
tick_positions = [0, 1, 2, 3, 4]  # adjust based on your actual data
ax.set_xticks(tick_positions)
ax.set_xticklabels(['$\Gamma$','$\Gamma$ & Rx$_1$','Rx$_1$','Rx$_1$ & Rx$_3$','Rx$_1$, Rx$_2$\n & Rx$_3$'])
sns.move_legend(
    ax, "lower center",
    bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False, columnspacing=0.5, handletextpad=0.3, borderaxespad=0.1
)
#plt.show()

In [14]:
import matplotlib.patches as patches

grouped_by_classifier = results_to_plot.groupby('Classifier')
plt.ion()
#plt.rc('text', usetex=True)
font_size = 50

colour_map = ['#00fb82', '#0000ff', '#b83a65']

rc = {
    'font.size': font_size,             # Base font size
    'axes.titlesize': font_size,        # Title
    'axes.labelsize': font_size,        # Axis labels
    'xtick.labelsize': font_size-15,     # X tick labels
    'ytick.labelsize': font_size,       # Y tick labels  
    'legend.fontsize': font_size,       # Legend
    'legend.title_fontsize': font_size, # Legend title
}

sort_dict = {'2 Rx':3,'Tx':0,'Tx and\n 1 Rx':2, '3 Rx':4, '1 Rx':2}
sns.set_theme(context="notebook", style='whitegrid', rc=rc)
x='S Parameter'
y='Precision'
hue='Type'

for name, df in grouped_by_classifier:
    plt.figure()
    ax = sns.barplot(df.sort_values(by=['S Parameter'], key=lambda x: x.map(sort_dict)), x=x, y=y, hue=hue, alpha=0.8, errorbar=None, palette=colour_map)
    ax.set(xlabel=None)
    ax.set_ylim(65,95)
    ax.set_ylabel('Accuracy (%)')
    
    # # ax.get_legend().remove()
    # rect = patches.Rectangle(
    #     (1.5, 64),  # (x, y) bottom-left corner
    #     2.95,  # width
    #     94.5-64,  # height
    #     linewidth=2,
    #     edgecolor='red',
    #     facecolor='none',
    #     linestyle='dashed',
    #     label='Fully\n Wireless'  # ✅ legend label
    # )
    # # Add a red dotted rectangle
    # ax.add_patch(rect)
    ax.legend()
    # tick_positions = [0, 1, 2, 3, 4]  # adjust based on your actual data
    # ax.set_xticks(tick_positions)
   # ax.set_xticklabels(['$\Gamma$','$\Gamma$ & Rx$_1$','Rx$_1$','Rx$_1$ & Rx$_3$','Rx$_1$, Rx$_2$\n & Rx$_3$'])
    sns.move_legend(
        ax, "lower center",
        bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False, columnspacing=0.5, handletextpad=0.3, borderaxespad=0.1
    )
    plt.show()

In [42]:
sns.move_legend(
    ax, "lower center",
    bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False, columnspacing=0.5, handletextpad=0.3, borderaxespad=0
)


In [80]:
from matplotlib.ticker import LinearLocator

ax.yaxis.set_major_locator(LinearLocator(numticks=4))  

# Data extraction logic 

In [ ]:
from vna.ml_model import get_full_results_df_from_classifier_pkls
results_for_retest = get_full_results_df_from_classifier_pkls(Path(r'D:\GLove Experiment 3'))
results_for_retest.to_pickle(r'C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Glove Gesture Experiment 3\classifier_results\reclassifciation_results_with_all_s_params.pkl')
classifier_results_path = Path(r"C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Old_Data_BackUp\Pickles\classifier_results")
PKL_RESULTS_FNAME = "classification_results_glove_experiment.pkl"

RESULTS_PATH_STRING = r"C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Glove Gesture Experiment 3"
results_folder_path = Path(RESULTS_PATH_STRING)
extracted_results_remove_8 = extract_remove_8_data(results_folder_path)

extracted_results_remove_8 = extracted_results_remove_8[extracted_results_remove_8['full or filtered'] == 'filtered']
extracted_results_remove_8['s_param'] = extracted_results_remove_8['s_param'].str.replace('_', ' ')
extracted_results_remove_8.columns = extracted_results_remove_8.columns.str.replace('_', ' ').str.title()
extracted_results_remove_8.to_pickle(r'C:\Users\2573758S\OneDrive - University of Glasgow\PhD\Experiments\Glove Gesture Experiment 3\classifier_results\extracted_results_glove_experiment_remove8_accuracy_filtered.pkl')